In [ ]:
import numpy as np

from matplotlib import pyplot as plt

## Display Case 1 Results

This notebook generates Figures 1, 2, 3, and 4 for Case 1 (ideal guide).

Before running this notebook, please execute the following script:

```
./run.sh
```

In [ ]:
plt.rcParams['font.size'] = 14
mmin = 0
mmax = 2.5
cwmin = -1.0
cwmax = 2.5

In [ ]:
def cross_section_index(direction, cross, x, y, z):
    '''
    Return indices of grid points corresponding to a specified cross-section.

    Parameters
    ----------
    direction : {'ew', 'ns', 'h'}
        Direction of the cross-section:
        'ew' : x–z section (fixed y),
        'ns' : y–z section (fixed x),
        'h'  : horizontal slice (fixed z).
    cross : float
        Target coordinate value. The nearest grid value is used.
    x, y, z : array_like
        Coordinates of the model grid.

    Returns
    -------
    idx : ndarray
        Indices of points belonging to the selected cross-section.

    Notes
    -----
    The cross-section is extracted at the grid location closest to
    the specified coordinate to ensure robustness against floating-point
    mismatch and non-uniform grids.
    '''
    direction = direction.lower()

    x0 = np.unique(x)
    y0 = np.unique(y)
    z0 = np.unique(z)

    def grid_spacing(arr, name):
        if len(arr) < 2:
            raise ValueError(f"{name} grid has insufficient points")
        return np.min(np.diff(arr))

    if direction == 'ew':
        # x(EW)-z cross section
        dy = grid_spacing(y0, "y")
        idx = np.where(np.abs(y - cross) < dy / 2.0)[0]

    elif direction == 'ns':
        # y(NS)-z cross section
        dx = grid_spacing(x0, "x")
        idx = np.where(np.abs(x - cross) < dx / 2.0)[0]

    elif direction == 'h':
        # horizontal slice
        dz = grid_spacing(z0, "z")
        idx = np.where(np.abs(z - cross) < dz / 2.0)[0]

    else:
        raise ValueError("direction must be one of {'ew', 'ns', 'h'}")

    if idx.size == 0:
        raise ValueError("No points found within the specified cross-section range")

    return idx

## Input anomaly (Figure 2)

In [ ]:
### Figure 2 ###

data = np.loadtxt('input_mag.in', delimiter='\t')

x = data[:, 0]
y = data[:, 1]
z = data[:, 2]
f = data[:, 3]

# --- グリッド軸 ---
x_unique = np.unique(x)
y_unique = np.unique(y)

nx = len(x_unique)
ny = len(y_unique)

# --- 並び替え（重要） ---
idx = np.lexsort((x, y))
x_sorted = x[idx]
y_sorted = y[idx]
f_sorted = f[idx]

# --- reshape ---
X = x_sorted.reshape(ny, nx)
Y = y_sorted.reshape(ny, nx)
F = f_sorted.reshape(ny, nx)

# --- plot ---
fig, ax = plt.subplots(figsize=(8, 8))

pc = ax.pcolormesh(X, Y, F, cmap='jet', shading='auto')
ax.contour(X, Y, F, levels=20, colors='k', linewidths=0.5)

ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')

cbar = plt.colorbar(pc, orientation='horizontal', shrink=0.5, pad=0.1)
cbar.set_label('(nT)')

plt.show()

## True model (Figure 1)

In [ ]:
res = np.loadtxt('true.model', delimiter='\t')

x = res[:, 0]
y = res[:, 1]
z = res[:, 2]
b0 = res[:, 3]

x_unique = np.unique(x)
y_unique = np.unique(y)
z_unique = np.unique(z)

nx = len(x_unique)
ny = len(y_unique)
nz = len(z_unique)

print(nx, ny, nz)

In [ ]:
### Figure 1 (b), (c) ###

yc0 = 0.0
yc = y_unique[np.argmin(np.abs(y_unique - yc0))]
idx1 = cross_section_index('EW', yc, x, y, z)

# sort（z -> x）
idx_sort = np.lexsort((x[idx1], z[idx1]))

xs = x[idx1][idx_sort]
zs = z[idx1][idx_sort]
bs = b0[idx1][idx_sort]

xx = xs.reshape(nz, nx)
zz = zs.reshape(nz, nx)
bb = bs.reshape(nz, nx)

# --- NS cross section ---
xc0 = 0.0
xc = y_unique[np.argmin(np.abs(x_unique - xc0))]
idx2 = cross_section_index('NS', xc, x, y, z)

# 並び替え（z → y）
idx_sort = np.lexsort((y[idx2], z[idx2]))

ys = y[idx2][idx_sort]
zs = z[idx2][idx_sort]
bs = b0[idx2][idx_sort]

yy = ys.reshape(nz, ny)
zz2 = zs.reshape(nz, ny)
bb2 = bs.reshape(nz, ny)

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EW
ax = axes[0]
pc = ax.pcolormesh(xx, zz, bb, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_aspect('equal')
ax.set_xlabel('y (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# NS
ax = axes[1]
pc = ax.pcolormesh(yy, zz2, bb2, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('z (km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

plt.show()

## Figure 3 and 4

In [ ]:
s = ['0', 'x', 'y', 'z']
data_dict = {}

for key in s:
    fname = f'weight_d{key}.vec'
    
    with open(fname, 'r') as f:
        lines = f.readlines()[2:]  # skip headers
    
    d = np.array([float(val.strip()) for val in lines])
    data_dict[key] = d

# extract
d0 = data_dict['0']
dx = data_dict['x']
dy = data_dict['y']
dz = data_dict['z']

res = np.loadtxt('model.data', delimiter='\t')
b = res[:, 3]

In [ ]:
### Figure 4 (a), (b) and Figure 3 ###

# =========================
# index of cross-section
# =========================
yc = y_unique[np.argmin(np.abs(y_unique - 0.0))]
idx1 = cross_section_index('EW', yc, x, y, z)

xc = x_unique[np.argmin(np.abs(x_unique - 0.0))]
idx2 = cross_section_index('NS', xc, x, y, z)

# =========================
#           EW
# =========================
idx_sort = np.lexsort((x[idx1], z[idx1]))

xs = x[idx1][idx_sort]
zs = z[idx1][idx_sort]

xx = xs.reshape(nz, nx)
zz = zs.reshape(nz, nx)

bb_ew = b[idx1][idx_sort].reshape(nz, nx)

# =========================
#           NS
# =========================
idx_sort = np.lexsort((y[idx2], z[idx2]))

ys = y[idx2][idx_sort]
zs = z[idx2][idx_sort]

yy = ys.reshape(nz, ny)
zz2 = zs.reshape(nz, ny)

bb_ns = b[idx2][idx_sort].reshape(nz, ny)

# =========================
#           model
# =========================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# EW
ax = axes[0]
pc = ax.pcolormesh(xx, -zz, bb_ew, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('y(km)')
ax.set_ylabel('z(km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# NS
ax = axes[1]
pc = ax.pcolormesh(yy, -zz2, bb_ns, cmap='jet', shading='auto')
pc.set_clim(mmin, mmax)
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(2.5, 0)
ax.set_aspect('equal')
ax.set_xlabel('x(km)')
ax.set_ylabel('z(km)')
plt.colorbar(pc, ax=ax, orientation='horizontal', shrink=0.5, pad=0.2, label='(A/m)')

# =========================
# weight（EW cross-section）
# =========================
def safe_log10(a):
    return np.log10(np.clip(a, 1e-20, None))

bb0 = d0[idx1][idx_sort].reshape(nz, nx)
bb1 = dx[idx1][idx_sort].reshape(nz, nx)
bb2 = dy[idx1][idx_sort].reshape(nz, nx)
bb3 = dz[idx1][idx_sort].reshape(nz, nx)

B = [bb0, bb1, bb2, bb3]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for i, ax in enumerate(axes.flat):
    pc = ax.pcolormesh(xx, -zz, safe_log10(B[i]), cmap='jet', shading='auto')

    if cwmax > 0:
        pc.set_clim(cwmin, cwmax)

    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(2.5, 0)
    ax.set_aspect('equal')
    ax.set_xlabel('y(km)')
    ax.set_ylabel('z(km)')

    plt.colorbar(pc, ax=ax, orientation='horizontal',
                 shrink=0.5, pad=0.2,
                 label=r'$\log_{10}(\mathbf{w})$')

plt.show()